# ALIGN-PARETO — M2: SFT from Qwen3-0.6B-Base
Turns the **unaligned** pretrained model into a helpful-only chat model by supervised fine-tuning:

| Data | Output |
|---|---|
| human-written replies: no_robots (~8.6k) + oasst2 best-ranked English (6k), refusal-like replies removed | `models/m2/sft_adapter` |

Every training reply ends with `<|endoftext|>` (`end_of_reply_token` in `configs/alignment.yaml`), so the model learns to stop.

**Kaggle setup:** input `align-pareto-src` attached (must contain `data/eval/eval_prompts.csv`), **GPU T4**, internet on
(the model and datasets come from the Hugging Face Hub; all public). Run from this machine with
`kaggle/run_on_kaggle.sh m2`. All logic lives in `src/`; this notebook only runs it and shows the results.

In [ ]:
# 1. Environment (one GPU: the trainers would otherwise wrap the model in DataParallel)
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import sys, json, platform, shutil, glob, subprocess
from pathlib import Path
import importlib.metadata as md

print("Python", platform.python_version())
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("No GPU: M2 needs one (Settings -> Accelerator -> GPU T4)")

In [ ]:
# 2. Dependencies (pinned to the versions the code was tested with)
PINNED = ["transformers==5.22.2", "trl==1.17.1", "peft==0.21.2", "datasets"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINNED], check=True)
# Kaggle preinstalls torchao 0.10; peft 0.21 refuses to wrap layers while an old torchao is importable (unused here)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
for pkg in ["torch", "transformers", "trl", "peft", "accelerate", "datasets", "huggingface_hub"]:
    try:
        print(f"{pkg:16s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:16s} MISSING")

In [ ]:
# 3. Project code: Kaggle dataset `align-pareto-src` -> /kaggle/working/align-pareto; locally, the repo itself
ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    hits = glob.glob("/kaggle/input/**/src/alignment/train_sft.py", recursive=True)
    assert hits, "align-pareto-src is missing or out of date (no src/alignment/train_sft.py): attach its latest version."
    ROOT = Path("/kaggle/working/align-pareto")
    shutil.copytree(Path(hits[0]).parents[2], ROOT, dirs_exist_ok=True)
else:
    ROOT = Path.cwd().parent
assert (ROOT / "data/eval/eval_prompts.csv").exists(), "data/eval/eval_prompts.csv missing: attach the latest align-pareto-src"
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
# 4. Run M2 SFT (QUICK=True: 3-step smoke test)
QUICK = False
from src.run_m2 import run_m2
metrics = run_m2(ROOT, quick=QUICK)
TAG = "m2_quick" if QUICK else "m2"
RES = ROOT / "results" / TAG
if ON_KAGGLE:  # keep the output small: raw downloads and optimizer checkpoints are not needed afterwards
    shutil.rmtree(ROOT / "data/raw/hf_cache", ignore_errors=True)
    for ckpt in (ROOT / "models" / TAG).glob("*_checkpoints"):
        shutil.rmtree(ckpt, ignore_errors=True)

In [ ]:
# 5. Training checks: SFT loss falls, the saved adapter reloads to the trained model
print(json.dumps({k: metrics[k] for k in ["sft", "reload_check"] if k in metrics}, indent=2))
from IPython.display import Image, display
if (RES / "plots/sft_training.png").exists():
    display(Image(filename=str(RES / "plots/sft_training.png"), width=900))

In [ ]:
# 6. Smoke test: raw base model vs SFT model, sampled with a fixed seed
# Pass: SFT share_stopped close to 1.0 and share_stray_chars close to 0
import pandas as pd
pd.set_option("display.max_colwidth", 300)
smoke = pd.read_csv(RES / "smoke_test.csv")
print(json.dumps(metrics["smoke_test"], indent=2))
q = metrics["smoke_test"]["sft"]
print("STOPPING OK" if q["share_stopped"] >= 0.9 and q["share_stray_chars"] <= 0.1 else "STOPPING STILL BROKEN: read the replies below")
display(smoke[["quadrant", "prompt", "reply_base_model", "reply_sft"]])

In [ ]:
# 7. One zip with everything M3 needs, for download from the Output panel
if ON_KAGGLE:
    subprocess.run(f"cd {ROOT} && zip -qr /kaggle/working/m2_outputs.zip results/{TAG} models/{TAG}/sft_adapter", shell=True, check=True)
    print("Saved /kaggle/working/m2_outputs.zip")

## Next: M3
Evaluates the SFT model (`models.load_sft_model(sft_adapter)`) against the raw base model.